# 架构设计：生成与计算分离

- 这种“vLLM 生成 + PyTorch 重算”的模式是目前开源社区（如 OpenRLHF）的主流最佳实践。

在使用 vLLM 时，流程变成了两步走：
- vLLM (推理引擎)：
  - 只负责“快”，生成文本序列 (generated_ids)，通常不输出完整的 Logits/Probs（或者为了显存效率不保存）。
- PyTorch Actor Model (训练引擎)：
  - 负责“算”，<font color='red'>拿着 vLLM 生成的 generated_ids 重新跑一次前向传播，计算 old_log_probs。</font>

## 背景：PPO 训练需要什么
在 RLHF（基于人类反馈的强化学习）中，**PPO 算法每一步需要这些数据：**

| 数据              | 含义                    | 用途            |
| :-------------- | :-------------------- | :------------ |
| `generated_ids` | 生成的文本序列               | 用来计算 reward   |
| `old_log_probs` | <font color='red'>**生成时**</font>每个 token 的对数概率 | PPO 的"旧策略"参考值 |
| `new_log_probs` | <font color='red'>**更新后**</font>每个 token 的对数概率 | PPO 的"新策略"当前值 |
| `advantages`    | 每个 token 的优势值         | 告诉模型哪些词该强化/抑制 |

### 核心矛盾：
- vLLM 生成很快，但它通常不输出/不保存 logits；<font color='red'>而计算 old_log_probs 又必须知道生成时每个 token 的 logits。</font>

## 分离架构的两步走
- 在使用 vLLM 时，流程变成了两步走：

### 第一步：vLLM 负责"生成"（快）

In [ ]:
输入 prompt: "请写一首关于春天的诗"

vLLM（推理引擎）
    ↓ 快速自回归生成
输出 generated_ids: [101, 2769, 4518, 3221, 884, ...]
对应文本: "春风又绿江南岸，明月何时照我还。"

#### vLLM 的特点：
- 用 PagedAttention、连续批处理、投机解码等技术，生成速度极快
- <font color='red'>为了省显存，生成完就丢弃中间 logits，只返回 token ID 序列</font>
  - **相当于一个"只写答案不写过程的超级速算器"**



In [ ]:
# 伪代码：vLLM 生成阶段
from vllm import LLM, SamplingParams

llm = LLM(model="your_model_path")
sampling_params = SamplingParams(temperature=0.7, max_tokens=512)

# prompts 是输入的问题列表
# outputs 包含了生成的 token_ids，但通常不包含完整的 log_probs
outputs = llm.generate(prompts, sampling_params)

# 提取生成的完整序列 ID: [Prompt + Response]
generated_ids_list = [output.token_ids for output in outputs]

### 第二步：PyTorch Actor Model 负责"计算"（准）
- vLLM 生成的数据需要转移回 PyTorch 的训练显存中。这里有一个极其重要的细节：Padding（填充）。
  - vLLM **生成的是不等长的序列。**
  - PyTorch 的 Actor Model 进行并行计算（Batch Forward）时，要求输入必须是矩形矩阵。
  - 因此，<font color='red'>必须将序列 Padding 到相同长度，并生成对应的 attention_mask</font>

In [ ]:
拿着 generated_ids: [101, 2769, 4518, 3221, 884, ...]

PyTorch Actor Model（训练引擎）
    ↓ 重新做一次完整的前向传播（forward）
输出 logits: [seq_len, vocab_size]

    ↓ 从 logits 中提取每个位置对应 token 的概率
    ↓ 取对数
得到 old_log_probs: [-0.5, -1.2, -0.8, -2.1, -0.3, ...]

#### <font color='red'>为什么能"重新算"？</font>
- 因为 generated_ids 是确定的序列。
  - **PyTorch Actor Model 把 [prompt_ids + generated_ids] 一起输入，模型会输出每个位置的 logits。然后对于 generated 部分的每个位置，取出实际生成的那个 token 对应的 logit，算 softmax 再取 log，就是 old_log_prob。**



In [ ]:
# 伪代码：Actor 模型计算 old_log_probs
import torch.nn.functional as F

# 1. 将 vLLM 生成的 IDs 转为 Tensor 并 Padding
input_ids = pad_sequences(generated_ids_list) 
attention_mask = create_mask(input_ids) # 创建掩码，忽略 padding 部分

# 2. 将数据搬到 Actor 模型的显卡上
input_ids = input_ids.to(actor_device)
attention_mask = attention_mask.to(actor_device)

# 3. 【关键】Actor 模型重新推理
# 注意：这里不需要梯度，因为只是为了获取基准概率
with torch.no_grad():
    logits = actor_model(input_ids, attention_mask=attention_mask).logits
    
    # 4. 计算 Log Softmax
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 5. 提取对应 Token 的概率 (即你图中的 gather 操作)
    # 这里的 input_ids 就是 vLLM 生成的 "标准答案"
    old_log_probs = torch.gather(
        log_probs, 
        dim=2, 
        index=input_ids.unsqueeze(-1)
    ).squeeze(-1)

## 完整流程图

In [1]:
┌─────────────────┐
│   Prompt 输入    │
└────────┬────────┘
         ▼
┌─────────────────┐
│  vLLM 推理引擎   │  ← 只负责"快"
│  (生成阶段)      │
│                 │
│  自回归生成       │
│  → generated_ids │
│  (丢弃 logits)   │
└────────┬────────┘
         ▼
┌─────────────────┐
│  计算 Reward     │  ← 用 reward model 打分
│  计算 Advantages │  ← 用 value model / GAE
└────────┬────────┘
         ▼
┌─────────────────┐
│ PyTorch Actor   │  ← 只负责"算"
│ Model (训练引擎) │
│                 │
│  重新 forward    │
│  prompt+response │
│  → logits       │
│  → old_log_probs │
└────────┬────────┘
         ▼
┌─────────────────┐
│   PPO 更新      │  ← 用 old_log_probs 和 new_log_probs
│  (多次 mini-batch│     计算 ratio、clipped loss、KL
│   梯度下降)      │
└─────────────────┘

SyntaxError: invalid character '┌' (U+250C) (2574196205.py, line 1)

## 用一个具体例子走一遍
假设词汇表只有 5 个词，prompt 是"今天天气"：
### Step 1: vLLM 生成


In [ ]:
vLLM 输入: [今天, 天气] 的 token IDs
vLLM 输出 generated_ids: [很好, 适合, 出门]  →  [7, 12, 33]

<font color='red'>vLLM 内部其实算过 logits，</font>比如：
- 位置1 logits: [很好:3.0, 不错:2.5, 糟糕:-1.0, ...] → 选了"很好"
- 位置2 logits: [适合:2.8, 可以:2.0, 不宜:-2.0, ...] → 选了"适合"
- 位置3 logits: [出门:3.5, 在家:1.2, 睡觉:0.5, ...] → 选了"出门"

<font color='red'>但 vLLM 为了速度，不保存这些 logits，只返回 [7, 12, 33] 

### Step 2: PyTorch Actor Model 重算

In [ ]:
输入完整序列: [今天, 天气, 很好, 适合, 出门]
            ↑prompt↑  ↑generated_ids↑

Actor Model 做 forward:
位置0 (今天): logits = [?, ?, ?, ?, ?]  → 不关心
位置1 (天气): logits = [?, ?, ?, ?, ?]  → 不关心
位置2 (很好): logits = [3.0, 2.5, -1.0, ...] 
              → 取出 "很好" 对应的 logit = 3.0
              → softmax → prob = 0.55
              → log(0.55) = -0.60  →  old_log_prob[0] = -0.60

位置3 (适合): logits = [2.8, 2.0, -2.0, ...]
              → 取出 "适合" 对应的 logit = 2.8
              → softmax → prob = 0.69
              → log(0.69) = -0.37  →  old_log_prob[1] = -0.37

位置4 (出门): logits = [3.5, 1.2, 0.5, ...]
              → 取出 "出门" 对应的 logit = 3.5
              → softmax → prob = 0.80
              → log(0.80) = -0.22  →  old_log_prob[2] = -0.22

#### 最终得到：
- old_log_probs = [-0.60, -0.37, -0.22]

## 为什么要分离？不直接让 vLLM 输出 logits 吗？

| 方案                       | 问题                                                                                                      |
| :----------------------- | :------------------------------------------------------------------------------------------------------ |
| **vLLM 输出 logits**       | vLLM 优化的是**推理吞吐**，每次生成只保留当前步的 logits，如果要保存全序列 logits，显存开销巨大（`seq_len × vocab_size × batch_size × fp16`） |
| **不用 vLLM，纯 PyTorch 生成** | <font color='red'>PyTorch 的生成效率远低于 vLLM，batch 大了 KV Cache 管理混乱，吞吐量差一个数量级 </font>                                                 |
| **分离架构**                 | **各取所长**：<font color='red'>vLLM 负责快速产出数据，PyTorch 负责精确计算梯度所需的中间量   </font>                                                         |

## 这种架构的代价
1. 重复计算：
  - 同样的序列，vLLM 生成时算过一次 logits，PyTorch 又算一次
    - 但 vLLM 的生成是推理优化过的（量化、投机解码等），PyTorch 的 forward 是训练精度的，**两者不完全等价**
    - 而且 vLLM 的生成阶段和 PyTorch 的计算阶段可以流水线并行
2. 数值一致性
  - ：<font color='red'>如果 vLLM 用了量化（INT8/INT4）而 PyTorch 是 FP16/BF16，两者算出的 logits 会有微小差异</font>
    - 解决方案：
      - vLLM 和 PyTorch Actor <font color='red'>用同样的权重精度</font>，<font color='blue'>或者 vLLM 只负责生成 ID，所有概率计算都交给 PyTorch</font>
  
## 一句话总结
- vLLM 是"快枪手"：只管飞速生成 token，不记过程；
- PyTorch Actor 是"审计员"：拿着生成的结果，重新算一遍账（logits → log_probs），供 PPO 训练使用。

### 这种"生成与计算分离"的本质，
- <font color='red'>是用两次前向传播的代价，换取推理速度和训练灵活性的平衡。</font>